# Searchis — FAISS IVFPQ Index Builder (GPU)

Builds the full FAISS index for ~3M arXiv documents on a free Colab T4 GPU.

**Before running:**
1. Upload `output-oai.json` (3.4 GB) to your Google Drive root (or update the path below)
2. Set runtime to **GPU**: Runtime → Change runtime type → T4 GPU
3. Run all cells — takes ~10–15 minutes

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Dependencies

In [ ]:
!pip install -q sentence-transformers faiss-gpu numpy

## 3. Verify GPU is Available

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## 4. Configure Paths

Update `INPUT_PATH` if your `output-oai.json` is in a different Drive folder.

In [ ]:
# --- CONFIGURE THESE ---
INPUT_PATH = "/content/drive/MyDrive/output-oai.json"  # Your sanitized dataset
OUTPUT_PATH = "/content/drive/MyDrive/searchis_index"   # Where to save the index files

# FAISS parameters (tuned for ~3M vectors)
D = 384          # Embedding dimensions (MiniLM-L6-v2)
M = 8            # PQ sub-quantizers
NBITS = 8        # Bits per sub-quantizer
NLIST = 1024     # IVF clusters (sqrt(3M) ≈ 1732, 1024 is good)
NPROBE = 10      # Clusters to search at query time
CHUNK_SIZE = 150000  # Documents per batch
BATCH_SIZE = 512     # Encoding batch size (GPU can handle more)

import os
assert os.path.exists(INPUT_PATH), f"File not found: {INPUT_PATH} — upload it to Drive first!"
print(f"Input file found: {os.path.getsize(INPUT_PATH) / 1e9:.1f} GB")

## 5. Build the FAISS Index

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import json
import gc
import time

# Load model on GPU (no ONNX — use PyTorch CUDA directly)
print("Loading SentenceTransformer model on GPU...")
model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
print("Model loaded.\n")

start_time = time.time()

# ============================================================
# STAGE 1: Gather training sample and train the IVFPQ quantizer
# ============================================================
print("=" * 60)
print("STAGE 1: Training IVFPQ quantizer")
print("=" * 60)

train_texts = []
sample_count = 0

with open(INPUT_PATH, "r") as infile:
    for line in infile:
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue
        full_text = record.get("clean_title", "") + ". " + record.get("clean_abstract", "")
        train_texts.append(full_text.strip())
        sample_count += 1
        if sample_count >= CHUNK_SIZE:
            break

print(f"Encoding training sample of {len(train_texts)} documents...")
train_embeddings = model.encode(
    train_texts,
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=BATCH_SIZE,
).astype(np.float32)

# Build and train the FAISS index on CPU, then move to GPU
quantizer = faiss.IndexFlatIP(D)
cpu_index = faiss.IndexIVFPQ(quantizer, D, NLIST, M, NBITS)

print("Training FAISS index on sample...")
cpu_index.train(train_embeddings)
print("Training complete.")

# Move to GPU for fast adding
res = faiss.StandardGpuResources()
gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

del train_texts, train_embeddings
gc.collect()
torch.cuda.empty_cache()

stage1_time = time.time() - start_time
print(f"\nStage 1 completed in {stage1_time:.1f}s\n")

In [ ]:
# ============================================================
# STAGE 2: Stream all documents and add to index
# ============================================================
print("=" * 60)
print("STAGE 2: Encoding & indexing all documents")
print("=" * 60)

stage2_start = time.time()
doc_ids = []
current_chunk_texts = []
total_indexed = 0

with open(INPUT_PATH, "r") as infile:
    for line in infile:
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue

        doc_ids.append(record.get("id"))
        full_text = record.get("clean_title", "") + ". " + record.get("clean_abstract", "")
        current_chunk_texts.append(full_text.strip())

        if len(current_chunk_texts) == CHUNK_SIZE:
            chunk_embeddings = model.encode(
                current_chunk_texts,
                show_progress_bar=False,
                normalize_embeddings=True,
                batch_size=BATCH_SIZE,
            ).astype(np.float32)

            gpu_index.add(chunk_embeddings)
            total_indexed += len(current_chunk_texts)

            elapsed = time.time() - stage2_start
            rate = total_indexed / elapsed
            est_remaining = (3_066_190 - total_indexed) / rate if rate > 0 else 0
            print(
                f"  ✓ {total_indexed:>10,} / ~3,066,190 docs "
                f"({total_indexed/3_066_190*100:5.1f}%) | "
                f"{rate:,.0f} docs/sec | "
                f"ETA: {est_remaining/60:.1f} min"
            )

            current_chunk_texts = []
            del chunk_embeddings
            gc.collect()

    # Process remaining
    if current_chunk_texts:
        chunk_embeddings = model.encode(
            current_chunk_texts,
            show_progress_bar=False,
            normalize_embeddings=True,
            batch_size=BATCH_SIZE,
        ).astype(np.float32)
        gpu_index.add(chunk_embeddings)
        total_indexed += len(current_chunk_texts)
        del chunk_embeddings
        gc.collect()

# Convert doc_ids to numpy for memory efficiency
doc_ids = np.array(doc_ids, dtype='U20')

stage2_time = time.time() - stage2_start
total_time = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"DONE! Indexed {total_indexed:,} documents")
print(f"Stage 1 (train):  {stage1_time:.1f}s")
print(f"Stage 2 (index):  {stage2_time:.1f}s")
print(f"Total:            {total_time:.1f}s ({total_time/60:.1f} min)")
print(f"{'=' * 60}")

## 6. Save Index to Google Drive

Saves two files:
- `searchis_index_faiss.index` — the FAISS IVFPQ index (~120–150 MB)
- `searchis_index_data.npz` — the doc_ids array (~60–80 MB)

These are compatible with your local `faiss_index.py` `load_from_disk()` method.

In [ ]:
print("Saving index to Google Drive...")

# Move index back to CPU for saving
cpu_index = faiss.index_gpu_to_cpu(gpu_index)

# Set nprobe so it's baked into the saved index
cpu_index.nprobe = NPROBE

# Save FAISS index
faiss.write_index(cpu_index, f"{OUTPUT_PATH}_faiss.index")
print(f"  ✓ Saved FAISS index: {os.path.getsize(f'{OUTPUT_PATH}_faiss.index') / 1e6:.1f} MB")

# Save doc_ids
np.savez_compressed(f"{OUTPUT_PATH}_data.npz", doc_ids=doc_ids)
print(f"  ✓ Saved doc_ids:     {os.path.getsize(f'{OUTPUT_PATH}_data.npz') / 1e6:.1f} MB")

print(f"\nFiles saved to Google Drive:")
print(f"  {OUTPUT_PATH}_faiss.index")
print(f"  {OUTPUT_PATH}_data.npz")
print(f"\nDownload these and place them in your local ./archive/ folder.")

## 7. Quick Sanity Check (Optional)

Run a test query to verify the index works before downloading.

In [ ]:
# Test search on GPU
test_query = "quantum computing error correction"

query_embedding = model.encode(
    [test_query], normalize_embeddings=True
).astype(np.float32)

gpu_index.nprobe = NPROBE
D_scores, I_indices = gpu_index.search(query_embedding, 10)

print(f"Test query: '{test_query}'")
print(f"\nTop 10 results:")
print(f"{'Rank':<6} {'Doc ID':<20} {'Score':<10}")
print("-" * 36)
for rank, (idx, score) in enumerate(zip(I_indices[0], D_scores[0]), 1):
    if idx != -1:
        print(f"{rank:<6} {doc_ids[idx]:<20} {score:<10.4f}")

## 8. Usage on Your Local Machine

After downloading the index files, load them locally:

```python
from faiss_index import VectorIndex

# Point to wherever you saved the files (without the _faiss.index / _data.npz suffix)
vector = VectorIndex()
vector.load_from_disk("./archive/searchis_index")

results = vector.search("quantum computing")
print(results)
```